In [7]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Telecom Customer Usage and Billing Analytics") \
    .getOrCreate()

In [8]:
%%writefile customers.csv
customer_id,customer_name,city,state,age,gender,plan_id,status
101,Rahul Sharma,Hyderabad,Telangana,35,Male,P101,Active
102,Priya Reddy,Bangalore,Karnataka,29,Female,P102,Active
103,Amit Kumar,Mumbai,Maharashtra,42,Male,P103,Inactive
104,Sneha Patel,Chennai,Tamil Nadu,31,Female,P101,Active
105,Farhan Ali,Delhi,Delhi,55,Male,P104,Active
106,Neha Singh,Pune,Maharashtra,38,Female,P102,Active
107,Arjun Verma,Hyderabad,Telangana,26,Male,P103,Inactive
108,Meera Nair,Kochi,Kerala,48,Female,P104,Active
109,Kiran Rao,Bangalore,Karnataka,33,Male,P101,Active
110,Nisha Reddy,Delhi,Delhi,41,Female,P102,Active
111,Ravi Kumar,Mumbai,Maharashtra,45,Male,P105,Active
112,Ayesha Khan,Hyderabad,Telangana,28,Female,,Active

Overwriting customers.csv


In [9]:
%%writefile usage.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1001,101,2026-01,45,900,120
1002,102,2026-01,30,600,80
1003,103,2026-01,12,250,40
1004,104,2026-01,55,1100,150
1005,105,2026-01,75,1500,200
1006,106,2026-01,28,500,60
1007,107,2026-01,10,200,20
1008,108,2026-01,80,1600,250
1009,109,2026-01,48,950,100
1010,110,2026-01,32,700,90
1011,120,2026-01,60,1300,140
1012,101,2026-02,50,1000,130
1013,102,2026-02,34,650,85
1014,104,2026-02,58,1200,160
1015,105,2026-02,,1450,210

Overwriting usage.csv


In [10]:
%%writefile plans.json
[
{
"plan_id": "P101",
"plan_name": "Smart Basic",
"monthly_fee": 499,
"data_limit_gb": 50,
"features": {
"unlimited_calls": true,
"ott_included": false,
"roaming": "National"
}
},
{
"plan_id": "P102",
"plan_name": "Smart Plus",
"monthly_fee": 799,
"data_limit_gb": 75,
"features": {
"unlimited_calls": true,
"ott_included": true,
"roaming": "National"
}
},
{
"plan_id": "P103",
"plan_name": "Budget Saver",
"monthly_fee": 299,
"data_limit_gb": 25,
"features": {
"unlimited_calls": false,
"ott_included": false,
"roaming": null
}
},
{
"plan_id": "P104",
"plan_name": "Premium Max",
"monthly_fee": 1199,
"data_limit_gb": 100,
"features": {
"unlimited_calls": true,
"ott_included": true,
"roaming": "International"
}
}
]

Overwriting plans.json


In [11]:
%%writefile payments.csv
payment_id,customer_id,bill_month,amount_paid,payment_mode,payment_status
5001,101,2026-01,499,UPI,Success
5002,102,2026-01,799,Card,Success
5003,103,2026-01,299,Cash,Failed
5004,104,2026-01,499,UPI,Success
5005,105,2026-01,1199,Card,Success
5006,106,2026-01,799,UPI,Success
5007,107,2026-01,299,Cash,Pending
5008,108,2026-01,1199,Card,Success
5009,109,2026-01,499,UPI,Success
5010,110,2026-01,799,UPI,Success
5011,112,2026-01,,UPI,Success
5012,101,2026-02,499,Card,Success
5013,102,2026-02,799,UPI,Success
5014,104,2026-02,499,UPI,Success
5015,105,2026-02,1199,,Pending

Overwriting payments.csv


In [12]:
# Part 1: Ingestion

customers_df = spark.read.csv("customers.csv", header=True, inferSchema=True)

In [13]:
usage_df = spark.read.csv("usage.csv", header=True, inferSchema=True)

In [14]:
payments_df = spark.read.csv("payments.csv", header=True, inferSchema=True)

In [15]:
plans_df = spark.read.option(
    "multiline",
    "true"
).json("plans.json")

In [16]:
customers_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)



In [17]:
usage_df.printSchema()

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)



In [18]:
payments_df.printSchema()

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



In [19]:
plans_df.printSchema()

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |    |-- roaming: string (nullable = true)
 |    |-- unlimited_calls: boolean (nullable = true)
 |-- monthly_fee: long (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- plan_name: string (nullable = true)



In [20]:
customers_df.count()

12

In [21]:
usage_df.count()

15

In [22]:
payments_df.count()

15

In [23]:
plans_df.count()

4

In [24]:
customers_df.write.mode("overwrite").parquet("bronze/customers")

In [25]:
usage_df.write.mode("overwrite").parquet("bronze/usage")

In [26]:
payments_df.write.mode("overwrite").parquet("bronze/payments")

In [27]:
plans_df.write.mode("overwrite").parquet("bronze/plans")

In [28]:
spark.read.parquet("bronze/customers").show()

spark.read.parquet("bronze/usage").show()

spark.read.parquet("bronze/payments").show()

spark.read.parquet("bronze/plans").show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [29]:
# Part 2: Data Cleaning

from pyspark.sql.functions import col, when

In [30]:
customers_df.filter(
    col("plan_id").isNull()
).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [31]:
usage_df.filter(
    col("data_used_gb").isNull()
).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [32]:
payments_df.filter(
    col("amount_paid").isNull()
).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [33]:
usage_clean_df = usage_df.fillna(
    {"data_used_gb": 0}
)

In [34]:
payments_clean_df = payments_df.fillna(
    {"amount_paid": 0}
)

In [35]:
payments_clean_df = payments_clean_df.fillna(
    {"payment_mode": "Not Provided"}
)

In [36]:
customers_clean_df = customers_df.fillna(
    {"plan_id": "UNKNOWN"}
)

In [37]:
customers_clean_df = customers_clean_df.withColumn(
    "data_quality_status",
    when(col("plan_id") == "UNKNOWN", "Issue")
    .otherwise("Valid")
)
customers_clean_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|              Valid|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|              Valid|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|              Valid|
|        108|   Meer

In [38]:
usage_clean_df = usage_clean_df.withColumn(
    "data_quality_status",
    when(col("data_used_gb") == 0, "Issue")
    .otherwise("Valid")
)
usage_clean_df.show()

+--------+-----------+-------------------+------------+------------+---------+-------------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+--------+-----------+-------------------+------------+------------+---------+-------------------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|              Valid|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|              Valid|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|              Valid|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|              Valid|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|              Valid|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|              Valid|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|              Valid|
|    1008|

In [39]:
payments_clean_df = payments_clean_df.withColumn(
    "data_quality_status",
    when(
        (col("amount_paid") == 0) |
        (col("payment_mode") == "Not Provided"),
        "Issue"
    ).otherwise("Valid")
)
payments_clean_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|              Valid|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|              Valid|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|              Valid|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|              Valid|
|      5007|        107|2026-01-01 00:00:00|        299

In [41]:
from pyspark.sql.functions import lit
plans_clean_df = plans_df.withColumn(
    "data_quality_status", lit("Valid")
)
plans_clean_df.show()

+-------------+--------------------+-----------+-------+------------+-------------------+
|data_limit_gb|            features|monthly_fee|plan_id|   plan_name|data_quality_status|
+-------------+--------------------+-----------+-------+------------+-------------------+
|           50|{false, National,...|        499|   P101| Smart Basic|              Valid|
|           75|{true, National, ...|        799|   P102|  Smart Plus|              Valid|
|           25|{false, NULL, false}|        299|   P103|Budget Saver|              Valid|
|          100|{true, Internatio...|       1199|   P104| Premium Max|              Valid|
+-------------+--------------------+-----------+-------+------------+-------------------+



In [42]:
customers_clean_df.write.mode("overwrite").parquet(
    "silver/customers"
)

In [43]:
usage_clean_df.write.mode("overwrite").parquet(
    "silver/usage"
)

In [44]:
payments_clean_df.write.mode("overwrite").parquet(
    "silver/payments"
)

In [45]:
plans_clean_df.write.mode("overwrite").parquet(
    "silver/plans"
)

In [46]:
spark.read.parquet("silver/customers").show()

spark.read.parquet("silver/usage").show()

spark.read.parquet("silver/payments").show()

spark.read.parquet("silver/plans").show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|              Valid|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|              Valid|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|              Valid|
|        108|   Meer

In [47]:
# Part 3: JSON Flattening

plans_flat_df = plans_clean_df.select(
    "plan_id",
    "plan_name",
    "monthly_fee",
    "data_limit_gb",
    "features.unlimited_calls",
    "features.ott_included",
    "features.roaming",
    "data_quality_status"
)
plans_flat_df.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|data_quality_status|
+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|              Valid|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|              Valid|
|   P103|Budget Saver|        299|           25|          false|       false|         NULL|              Valid|
|   P104| Premium Max|       1199|          100|           true|        true|International|              Valid|
+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+



In [48]:
plans_flat_df.select(
    "plan_id",
    "plan_name",
    "unlimited_calls"
).show()

+-------+------------+---------------+
|plan_id|   plan_name|unlimited_calls|
+-------+------------+---------------+
|   P101| Smart Basic|           true|
|   P102|  Smart Plus|           true|
|   P103|Budget Saver|          false|
|   P104| Premium Max|           true|
+-------+------------+---------------+



In [49]:
plans_flat_df.select(
    "plan_id",
    "plan_name",
    "ott_included"
).show()

+-------+------------+------------+
|plan_id|   plan_name|ott_included|
+-------+------------+------------+
|   P101| Smart Basic|       false|
|   P102|  Smart Plus|        true|
|   P103|Budget Saver|       false|
|   P104| Premium Max|        true|
+-------+------------+------------+



In [50]:
plans_flat_df.select(
    "plan_id",
    "plan_name",
    "roaming"
).show()

+-------+------------+-------------+
|plan_id|   plan_name|      roaming|
+-------+------------+-------------+
|   P101| Smart Basic|     National|
|   P102|  Smart Plus|     National|
|   P103|Budget Saver|         NULL|
|   P104| Premium Max|International|
+-------+------------+-------------+



In [51]:
plans_flat_df = plans_flat_df.fillna(
    {"roaming": "Not Available"}
)
plans_flat_df.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|data_quality_status|
+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|              Valid|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|              Valid|
|   P103|Budget Saver|        299|           25|          false|       false|Not Available|              Valid|
|   P104| Premium Max|       1199|          100|           true|        true|International|              Valid|
+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+



In [53]:
plans_flat_df.write.mode("overwrite").parquet(
    "silver/plans_flat"
)
spark.read.parquet(
    "silver/plans_flat"
).show(truncate=False)

+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |data_quality_status|
+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |Valid              |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |Valid              |
|P103   |Budget Saver|299        |25           |false          |false       |Not Available|Valid              |
|P104   |Premium Max |1199       |100          |true           |true        |International|Valid              |
+-------+------------+-----------+-------------+---------------+------------+-------------+-------------------+



In [54]:
# Part 4: Joins

customers_plans_df = customers_clean_df.join(
    plans_flat_df,
    on="plan_id",
    how="left"
)
customers_plans_df.show()

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|data_quality_status|
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+
|   P101|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active|              Valid| Smart Basic|        499|           50|           true|       false|     National|              Valid|
|   P102|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|  Active|              Valid|  Smart Plus|        799|           75|           true|        true|     National|              Valid|


In [55]:
customers_usage_df = customers_clean_df.join(
    usage_clean_df,
    on="customer_id",
    how="left"
)
customers_usage_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|    1012|2026-02-01 00:00:00|          50|        1000|      130|              Valid|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|    1001|2026-01-01 00:00:00|          45|         900|      120|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|        

In [56]:
customers_payments_df = customers_clean_df.join(
    payments_clean_df,
    on="customer_id",
    how="left"
)
customers_payments_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|      5012|2026-02-01 00:00:00|        499|        Card|       Success|              Valid|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|      5001|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Fe

In [57]:
customer_usage_billing_df = customers_clean_df \
    .join(plans_flat_df, on="plan_id", how="left") \
    .join(usage_clean_df, on="customer_id", how="left") \
    .join(payments_clean_df, on="customer_id", how="left")
customer_usage_billing_df.show(truncate=False)

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|plan_id|customer_name|city     |state      |age|gender|status  |data_quality_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |data_quality_status|usage_id|usage_month        |data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|bill_month         |amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+--------+-------------------+------------+-------

In [58]:
customers_clean_df.join(
    plans_flat_df,
    on="plan_id",
    how="left_anti"
).show()

+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|data_quality_status|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|              Valid|
|UNKNOWN|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|              Issue|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+



In [59]:
usage_clean_df.join(
    customers_clean_df,
    on="customer_id",
    how="left_anti"
).show()

+-----------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|              Valid|
+-----------+--------+-------------------+------------+------------+---------+-------------------+



In [60]:
payments_clean_df.join(
    customers_clean_df,
    on="customer_id",
    how="left_anti"
).show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



In [61]:
# Part 5: Transformations

customer_usage_billing_df = customer_usage_billing_df.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70, "Heavy User")
    .when(col("data_used_gb") >= 30, "Medium User")
    .otherwise("Low User")
)
customer_usage_billing_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+--------+----------

In [62]:
customer_usage_billing_df = customer_usage_billing_df.withColumn(
    "payment_category",
    when(col("amount_paid") >= 1000, "High Payment")
    .when(col("amount_paid") >= 500, "Medium Payment")
    .otherwise("Low Payment")
)
customer_usage_billing_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-----

In [63]:
customer_usage_billing_df = customer_usage_billing_df.withColumn(
    "churn_risk",
    when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success"),
        "High Risk"
    )
    .when(col("data_used_gb") < 15, "Medium Risk")
    .otherwise("Low Risk")
)
customer_usage_billing_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+--------

In [64]:
customer_usage_billing_df = customer_usage_billing_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)
customer_usage_billing_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+----------

In [65]:
customer_usage_billing_df = customer_usage_billing_df.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0, "Yes")
    .otherwise("No")
)
customer_usage_billing_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+-------------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+---

In [66]:
# Part 6: Aggregations

customer_usage_full_df = customers_clean_df \
    .join(plans_flat_df, on="plan_id", how="left") \
    .join(usage_clean_df, on="customer_id", how="left")

customer_payments_full_df = customers_clean_df \
    .join(plans_flat_df, on="plan_id", how="left") \
    .join(payments_clean_df, on="customer_id", how="left")

In [67]:
from pyspark.sql.functions import sum, avg, countDistinct

In [68]:
customers_clean_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [69]:
customers_clean_df.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    2|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    2|
|  Telangana|    3|
|Maharashtra|    3|
+-----------+-----+



In [70]:
customers_clean_df.groupBy("plan_id").count().show()

+-------+-----+
|plan_id|count|
+-------+-----+
|   P105|    1|
|   P102|    3|
|UNKNOWN|    1|
|   P103|    2|
|   P104|    2|
|   P101|    3|
+-------+-----+



In [71]:
customer_usage_billing_df.groupBy("usage_category") \
    .agg(countDistinct("customer_id").alias("customer_count")) \
    .show()

+--------------+--------------+
|usage_category|customer_count|
+--------------+--------------+
|   Medium User|             5|
|    Heavy User|             2|
|      Low User|             6|
+--------------+--------------+



In [72]:
customer_usage_billing_df.groupBy("churn_risk") \
    .agg(countDistinct("customer_id").alias("customer_count")) \
    .show()

+-----------+--------------+
| churn_risk|customer_count|
+-----------+--------------+
|   Low Risk|            10|
|Medium Risk|             1|
|  High Risk|             3|
+-----------+--------------+



In [73]:
customer_usage_full_df.groupBy("plan_id", "plan_name") \
    .agg(sum("data_used_gb").alias("total_data_usage")) \
    .show()

+-------+------------+----------------+
|plan_id|   plan_name|total_data_usage|
+-------+------------+----------------+
|   P105|        NULL|            NULL|
|   P103|Budget Saver|              22|
|   P101| Smart Basic|             256|
|   P104| Premium Max|             155|
|UNKNOWN|        NULL|            NULL|
|   P102|  Smart Plus|             124|
+-------+------------+----------------+



In [74]:
customer_usage_full_df.groupBy("plan_id", "plan_name") \
    .agg(avg("data_used_gb").alias("avg_data_usage")) \
    .show()

+-------+------------+------------------+
|plan_id|   plan_name|    avg_data_usage|
+-------+------------+------------------+
|   P105|        NULL|              NULL|
|   P103|Budget Saver|              11.0|
|   P101| Smart Basic|              51.2|
|   P104| Premium Max|51.666666666666664|
|UNKNOWN|        NULL|              NULL|
|   P102|  Smart Plus|              31.0|
+-------+------------+------------------+



In [75]:
customer_usage_full_df.groupBy("city") \
    .agg(sum("call_minutes").alias("total_call_minutes")) \
    .show()

+---------+------------------+
|     city|total_call_minutes|
+---------+------------------+
|Bangalore|              2200|
|    Kochi|              1600|
|  Chennai|              2300|
|   Mumbai|               250|
|     Pune|               500|
|    Delhi|              3650|
|Hyderabad|              2100|
+---------+------------------+



In [76]:
customer_usage_full_df.groupBy("state") \
    .agg(sum("sms_count").alias("total_sms_count")) \
    .show()

+-----------+---------------+
|      state|total_sms_count|
+-----------+---------------+
|  Karnataka|            265|
|     Kerala|            250|
| Tamil Nadu|            310|
|      Delhi|            500|
|  Telangana|            270|
|Maharashtra|            100|
+-----------+---------------+



In [77]:
customer_payments_full_df.filter(col("payment_status") == "Success") \
    .agg(sum("amount_paid").alias("total_successful_revenue")) \
    .show()

+------------------------+
|total_successful_revenue|
+------------------------+
|                    8089|
+------------------------+



In [78]:
customer_payments_full_df.filter(col("payment_status") == "Success") \
    .groupBy("city") \
    .agg(sum("amount_paid").alias("revenue")) \
    .show()

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|   2097|
|    Kochi|   1199|
|  Chennai|    998|
|     Pune|    799|
|    Delhi|   1998|
|Hyderabad|    998|
+---------+-------+



In [79]:
customer_payments_full_df.filter(col("payment_status") == "Success") \
    .groupBy("plan_id", "plan_name") \
    .agg(sum("amount_paid").alias("revenue")) \
    .show()

+-------+-----------+-------+
|plan_id|  plan_name|revenue|
+-------+-----------+-------+
|   P101|Smart Basic|   2495|
|   P104|Premium Max|   2398|
|UNKNOWN|       NULL|      0|
|   P102| Smart Plus|   3196|
+-------+-----------+-------+



In [80]:
customer_payments_full_df.filter(col("payment_status") == "Success") \
    .groupBy("payment_mode") \
    .agg(sum("amount_paid").alias("revenue")) \
    .show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|        Card|   3696|
|         UPI|   4393|
+------------+-------+



In [81]:
customer_payments_full_df.filter(col("payment_status") == "Success") \
    .groupBy("plan_id", "plan_name") \
    .agg(sum("amount_paid").alias("revenue")) \
    .orderBy(col("revenue").desc()) \
    .show(1)

+-------+----------+-------+
|plan_id| plan_name|revenue|
+-------+----------+-------+
|   P102|Smart Plus|   3196|
+-------+----------+-------+
only showing top 1 row


In [82]:
customer_payments_full_df.filter(col("payment_status") == "Success") \
    .groupBy("city") \
    .agg(sum("amount_paid").alias("revenue")) \
    .orderBy(col("revenue").desc()) \
    .show(1)

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|   2097|
+---------+-------+
only showing top 1 row


In [83]:
# Part 7: Window Functions

from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number, lag, lead, sum, col

In [84]:
customer_total_usage_df = customer_usage_full_df.groupBy(
    "customer_id", "customer_name"
).agg(sum("data_used_gb").alias("total_data_used"))

window_spec = Window.orderBy(col("total_data_used").desc())
customer_total_usage_df = customer_total_usage_df.withColumn(
    "rank", rank().over(window_spec)
)
customer_total_usage_df.show()

+-----------+-------------+---------------+----+
|customer_id|customer_name|total_data_used|rank|
+-----------+-------------+---------------+----+
|        104|  Sneha Patel|            113|   1|
|        101| Rahul Sharma|             95|   2|
|        108|   Meera Nair|             80|   3|
|        105|   Farhan Ali|             75|   4|
|        102|  Priya Reddy|             64|   5|
|        109|    Kiran Rao|             48|   6|
|        110|  Nisha Reddy|             32|   7|
|        106|   Neha Singh|             28|   8|
|        103|   Amit Kumar|             12|   9|
|        107|  Arjun Verma|             10|  10|
|        112|  Ayesha Khan|           NULL|  11|
|        111|   Ravi Kumar|           NULL|  11|
+-----------+-------------+---------------+----+



In [85]:
customer_total_revenue_df = customer_payments_full_df.filter(
    col("payment_status") == "Success"
).groupBy("customer_id", "customer_name").agg(
    sum("amount_paid").alias("total_amount_paid")
)

window_spec = Window.orderBy(col("total_amount_paid").desc())
customer_total_revenue_df = customer_total_revenue_df.withColumn(
    "rank", rank().over(window_spec)
)
customer_total_revenue_df.show()

+-----------+-------------+-----------------+----+
|customer_id|customer_name|total_amount_paid|rank|
+-----------+-------------+-----------------+----+
|        102|  Priya Reddy|             1598|   1|
|        108|   Meera Nair|             1199|   2|
|        105|   Farhan Ali|             1199|   2|
|        101| Rahul Sharma|              998|   4|
|        104|  Sneha Patel|              998|   4|
|        110|  Nisha Reddy|              799|   6|
|        106|   Neha Singh|              799|   6|
|        109|    Kiran Rao|              499|   8|
|        112|  Ayesha Khan|                0|   9|
+-----------+-------------+-----------------+----+



In [86]:
window_spec = Window.orderBy(col("total_data_used").desc())
top3_data_users_df = customer_total_usage_df.withColumn(
    "row_num", row_number().over(window_spec)
)
top3_data_users_df.filter(col("row_num") <= 3).show()

+-----------+-------------+---------------+----+-------+
|customer_id|customer_name|total_data_used|rank|row_num|
+-----------+-------------+---------------+----+-------+
|        104|  Sneha Patel|            113|   1|      1|
|        101| Rahul Sharma|             95|   2|      2|
|        108|   Meera Nair|             80|   3|      3|
+-----------+-------------+---------------+----+-------+



In [87]:
window_spec = Window.orderBy(col("total_amount_paid").desc())
top3_revenue_customers_df = customer_total_revenue_df.withColumn(
    "row_num", row_number().over(window_spec)
)
top3_revenue_customers_df.filter(col("row_num") <= 3).show()

+-----------+-------------+-----------------+----+-------+
|customer_id|customer_name|total_amount_paid|rank|row_num|
+-----------+-------------+-----------------+----+-------+
|        102|  Priya Reddy|             1598|   1|      1|
|        105|   Farhan Ali|             1199|   2|      2|
|        108|   Meera Nair|             1199|   2|      3|
+-----------+-------------+-----------------+----+-------+



In [88]:
customer_city_revenue_df = customer_payments_full_df.filter(
    col("payment_status") == "Success"
).groupBy("city", "customer_id", "customer_name").agg(
    sum("amount_paid").alias("total_amount_paid")
)

window_spec = Window.partitionBy("city").orderBy(col("total_amount_paid").desc())
top_customer_by_city_df = customer_city_revenue_df.withColumn(
    "row_num", row_number().over(window_spec)
)
top_customer_by_city_df.filter(col("row_num") == 1).show()

+---------+-----------+-------------+-----------------+-------+
|     city|customer_id|customer_name|total_amount_paid|row_num|
+---------+-----------+-------------+-----------------+-------+
|Bangalore|        102|  Priya Reddy|             1598|      1|
|  Chennai|        104|  Sneha Patel|              998|      1|
|    Delhi|        105|   Farhan Ali|             1199|      1|
|Hyderabad|        101| Rahul Sharma|              998|      1|
|    Kochi|        108|   Meera Nair|             1199|      1|
|     Pune|        106|   Neha Singh|              799|      1|
+---------+-----------+-------------+-----------------+-------+



In [89]:
customer_plan_revenue_df = customer_payments_full_df.filter(
    col("payment_status") == "Success"
).groupBy("plan_id", "plan_name", "customer_id", "customer_name").agg(
    sum("amount_paid").alias("total_amount_paid")
)

window_spec = Window.partitionBy("plan_id").orderBy(col("total_amount_paid").desc())
top_customer_by_plan_df = customer_plan_revenue_df.withColumn(
    "row_num", row_number().over(window_spec)
)
top_customer_by_plan_df.filter(col("row_num") == 1).show()

+-------+-----------+-----------+-------------+-----------------+-------+
|plan_id|  plan_name|customer_id|customer_name|total_amount_paid|row_num|
+-------+-----------+-----------+-------------+-----------------+-------+
|   P101|Smart Basic|        104|  Sneha Patel|              998|      1|
|   P102| Smart Plus|        102|  Priya Reddy|             1598|      1|
|   P104|Premium Max|        108|   Meera Nair|             1199|      1|
|UNKNOWN|       NULL|        112|  Ayesha Khan|                0|      1|
+-------+-----------+-----------+-------------+-----------------+-------+



In [90]:
monthly_revenue_df = customer_payments_full_df.filter(
    col("payment_status") == "Success"
).groupBy("bill_month").agg(
    sum("amount_paid").alias("monthly_revenue")
)

window_spec = Window.orderBy("bill_month")
monthly_revenue_df = monthly_revenue_df.withColumn(
    "running_total", sum("monthly_revenue").over(window_spec)
)
monthly_revenue_df.show()

+-------------------+---------------+-------------+
|         bill_month|monthly_revenue|running_total|
+-------------------+---------------+-------------+
|2026-01-01 00:00:00|           6292|         6292|
|2026-02-01 00:00:00|           1797|         8089|
+-------------------+---------------+-------------+



In [91]:
window_spec = Window.partitionBy("customer_id").orderBy("usage_month")

usage_lag_df = customer_usage_full_df.withColumn(
    "previous_month_usage", lag("data_used_gb", 1).over(window_spec)
)
usage_lag_df.select(
    "customer_id", "customer_name", "usage_month",
    "data_used_gb", "previous_month_usage"
).show()

+-----------+-------------+-------------------+------------+--------------------+
|customer_id|customer_name|        usage_month|data_used_gb|previous_month_usage|
+-----------+-------------+-------------------+------------+--------------------+
|        101| Rahul Sharma|2026-01-01 00:00:00|          45|                NULL|
|        101| Rahul Sharma|2026-02-01 00:00:00|          50|                  45|
|        102|  Priya Reddy|2026-01-01 00:00:00|          30|                NULL|
|        102|  Priya Reddy|2026-02-01 00:00:00|          34|                  30|
|        103|   Amit Kumar|2026-01-01 00:00:00|          12|                NULL|
|        104|  Sneha Patel|2026-01-01 00:00:00|          55|                NULL|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|                  55|
|        105|   Farhan Ali|2026-01-01 00:00:00|          75|                NULL|
|        105|   Farhan Ali|2026-02-01 00:00:00|           0|                  75|
|        106|   

In [92]:
window_spec = Window.partitionBy("customer_id").orderBy("usage_month")

usage_lead_df = customer_usage_full_df.withColumn(
    "next_month_usage", lead("data_used_gb", 1).over(window_spec)
)
usage_lead_df.select(
    "customer_id", "customer_name", "usage_month",
    "data_used_gb", "next_month_usage"
).show()

+-----------+-------------+-------------------+------------+----------------+
|customer_id|customer_name|        usage_month|data_used_gb|next_month_usage|
+-----------+-------------+-------------------+------------+----------------+
|        101| Rahul Sharma|2026-01-01 00:00:00|          45|              50|
|        101| Rahul Sharma|2026-02-01 00:00:00|          50|            NULL|
|        102|  Priya Reddy|2026-01-01 00:00:00|          30|              34|
|        102|  Priya Reddy|2026-02-01 00:00:00|          34|            NULL|
|        103|   Amit Kumar|2026-01-01 00:00:00|          12|            NULL|
|        104|  Sneha Patel|2026-01-01 00:00:00|          55|              58|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|            NULL|
|        105|   Farhan Ali|2026-01-01 00:00:00|          75|               0|
|        105|   Farhan Ali|2026-02-01 00:00:00|           0|            NULL|
|        106|   Neha Singh|2026-01-01 00:00:00|          28|    

In [93]:
usage_lag_df.filter(
    col("data_used_gb") > col("previous_month_usage")
).select(
    "customer_id", "customer_name", "usage_month",
    "data_used_gb", "previous_month_usage"
).show()

+-----------+-------------+-------------------+------------+--------------------+
|customer_id|customer_name|        usage_month|data_used_gb|previous_month_usage|
+-----------+-------------+-------------------+------------+--------------------+
|        101| Rahul Sharma|2026-02-01 00:00:00|          50|                  45|
|        102|  Priya Reddy|2026-02-01 00:00:00|          34|                  30|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|                  55|
+-----------+-------------+-------------------+------------+--------------------+



In [94]:
# Part 8: Spark SQL

customers_clean_df.createOrReplaceTempView("customers")
usage_clean_df.createOrReplaceTempView("usage")
payments_clean_df.createOrReplaceTempView("payments")
plans_flat_df.createOrReplaceTempView("plans")

In [95]:
spark.sql("""
SELECT *
FROM customers
WHERE status = 'Active'
""").show()

+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|Active|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|Active|              Valid|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|Active|              Valid|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|Active|              Valid|
|        110|  Nisha Reddy|    Delhi|   

In [96]:
spark.sql("""
SELECT city, COUNT(*) AS customer_count
FROM customers
GROUP BY city
""").show()

+---------+--------------+
|     city|customer_count|
+---------+--------------+
|Bangalore|             2|
|    Kochi|             1|
|  Chennai|             1|
|   Mumbai|             2|
|     Pune|             1|
|    Delhi|             2|
|Hyderabad|             3|
+---------+--------------+



In [97]:
spark.sql("""
SELECT pl.plan_id, pl.plan_name,
       SUM(p.amount_paid) AS revenue
FROM payments p
JOIN customers c ON p.customer_id = c.customer_id
JOIN plans pl ON c.plan_id = pl.plan_id
WHERE p.payment_status = 'Success'
GROUP BY pl.plan_id, pl.plan_name
ORDER BY revenue DESC
""").show()

+-------+-----------+-------+
|plan_id|  plan_name|revenue|
+-------+-----------+-------+
|   P102| Smart Plus|   3196|
|   P101|Smart Basic|   2495|
|   P104|Premium Max|   2398|
+-------+-----------+-------+



In [98]:
spark.sql("""
SELECT u.customer_id, c.customer_name, u.usage_month, u.data_used_gb
FROM usage u
JOIN customers c ON u.customer_id = c.customer_id
WHERE u.data_used_gb >= 70
""").show()

+-----------+-------------+-------------------+------------+
|customer_id|customer_name|        usage_month|data_used_gb|
+-----------+-------------+-------------------+------------+
|        105|   Farhan Ali|2026-01-01 00:00:00|          75|
|        108|   Meera Nair|2026-01-01 00:00:00|          80|
+-----------+-------------+-------------------+------------+



In [99]:
spark.sql("""
SELECT DISTINCT c.customer_id, c.customer_name, c.status, p.payment_status
FROM customers c
LEFT JOIN payments p ON c.customer_id = p.customer_id
WHERE c.status = 'Inactive'
   OR p.payment_status != 'Success'
   OR p.payment_status IS NULL
""").show()

+-----------+-------------+--------+--------------+
|customer_id|customer_name|  status|payment_status|
+-----------+-------------+--------+--------------+
|        105|   Farhan Ali|  Active|       Pending|
|        107|  Arjun Verma|Inactive|       Pending|
|        103|   Amit Kumar|Inactive|        Failed|
|        111|   Ravi Kumar|  Active|          NULL|
+-----------+-------------+--------+--------------+



In [100]:
spark.sql("""
SELECT c.*
FROM customers c
LEFT JOIN plans pl ON c.plan_id = pl.plan_id
WHERE pl.plan_id IS NULL
""").show()

+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|   P105|Active|              Valid|
|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|UNKNOWN|Active|              Issue|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+



In [101]:
spark.sql("""
SELECT *
FROM payments
WHERE payment_status IN ('Failed', 'Pending')
""").show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|              Valid|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|              Valid|
|      5015|        105|2026-02-01 00:00:00|       1199|Not Provided|       Pending|              Issue|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+



In [102]:
spark.sql("""
SELECT c.customer_id, c.customer_name,
       SUM(u.data_used_gb) AS total_data_used
FROM usage u
JOIN customers c ON u.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_data_used DESC
LIMIT 5
""").show()

+-----------+-------------+---------------+
|customer_id|customer_name|total_data_used|
+-----------+-------------+---------------+
|        104|  Sneha Patel|            113|
|        101| Rahul Sharma|             95|
|        108|   Meera Nair|             80|
|        105|   Farhan Ali|             75|
|        102|  Priya Reddy|             64|
+-----------+-------------+---------------+



In [103]:
spark.sql("""
SELECT payment_mode,
       SUM(amount_paid) AS revenue
FROM payments
WHERE payment_status = 'Success'
GROUP BY payment_mode
ORDER BY revenue DESC
""").show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|         UPI|   4393|
|        Card|   3696|
+------------+-------+



In [106]:
# Part 9: Full Refresh and Incremental Load

customer_usage_billing_df = customers_clean_df.withColumnRenamed(
        "data_quality_status", "customer_dq_status"
    ) \
    .join(
        plans_flat_df.withColumnRenamed("data_quality_status", "plan_dq_status"),
        on="plan_id", how="left"
    ) \
    .join(
        usage_clean_df.withColumnRenamed("data_quality_status", "usage_dq_status"),
        on="customer_id", how="left"
    ) \
    .join(
        payments_clean_df.withColumnRenamed("data_quality_status", "payment_dq_status"),
        on="customer_id", how="left"
    )
customer_usage_billing_df.show(truncate=False)

+-----------+-------+-------------+---------+-----------+---+------+--------+------------------+------------+-----------+-------------+---------------+------------+-------------+--------------+--------+-------------------+------------+------------+---------+---------------+----------+-------------------+-----------+------------+--------------+-----------------+
|customer_id|plan_id|customer_name|city     |state      |age|gender|status  |customer_dq_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |plan_dq_status|usage_id|usage_month        |data_used_gb|call_minutes|sms_count|usage_dq_status|payment_id|bill_month         |amount_paid|payment_mode|payment_status|payment_dq_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------------+------------+-----------+-------------+---------------+------------+-------------+--------------+--------+-------------------+------------+------------+---------+--------------

In [107]:
gold_path = "/content/gold_usage_billing"

customer_usage_billing_df.write.mode("overwrite").parquet(gold_path)

In [108]:
customer_usage_billing_df.write.mode("overwrite") \
    .partitionBy("usage_month") \
    .parquet(gold_path)

In [109]:
%%writefile incremental_usage_march.csv
usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-03,52,1050,135
1017,102,2026-03,38,700,90
1018,104,2026-03,60,1250,170
1019,105,2026-03,78,1550,220
1020,108,2026-03,85,1700,260

Writing incremental_usage_march.csv


In [110]:
incremental_usage_df = spark.read.csv(
    "incremental_usage_march.csv", header=True, inferSchema=True
)

incremental_usage_df.write.mode("overwrite").parquet("/content/incremental_usage_march")

In [111]:
inc_usage_df = spark.read.parquet("/content/incremental_usage_march")
inc_usage_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1016|        101|2026-03-01 00:00:00|          52|        1050|      135|
|    1017|        102|2026-03-01 00:00:00|          38|         700|       90|
|    1018|        104|2026-03-01 00:00:00|          60|        1250|      170|
|    1019|        105|2026-03-01 00:00:00|          78|        1550|      220|
|    1020|        108|2026-03-01 00:00:00|          85|        1700|      260|
+--------+-----------+-------------------+------------+------------+---------+



In [112]:
inc_usage_clean_df = inc_usage_df.fillna(
    {"data_used_gb": 0}
)
inc_usage_clean_df = inc_usage_clean_df.withColumn(
    "data_quality_status",
    when(col("data_used_gb") == 0, "Issue")
    .otherwise("Valid")
)
inc_usage_clean_df.write.mode("append").parquet("silver/usage")

In [113]:
updated_usage_df = spark.read.parquet("silver/usage")
updated_customer_usage_full_df = customers_clean_df.withColumnRenamed(
        "data_quality_status", "customer_dq_status"
    ) \
    .join(
        plans_flat_df.withColumnRenamed("data_quality_status", "plan_dq_status"),
        on="plan_id", how="left"
    ) \
    .join(
        updated_usage_df.withColumnRenamed("data_quality_status", "usage_dq_status"),
        on="customer_id", how="left"
    )
customer_usage_summary_df = updated_customer_usage_full_df.groupBy(
    "customer_id", "customer_name"
).agg(sum("data_used_gb").alias("total_data_used"))
customer_usage_summary_df.show()

+-----------+-------------+---------------+
|customer_id|customer_name|total_data_used|
+-----------+-------------+---------------+
|        107|  Arjun Verma|             10|
|        108|   Meera Nair|            165|
|        109|    Kiran Rao|             48|
|        110|  Nisha Reddy|             32|
|        105|   Farhan Ali|            153|
|        101| Rahul Sharma|            147|
|        112|  Ayesha Khan|           NULL|
|        104|  Sneha Patel|            173|
|        103|   Amit Kumar|             12|
|        106|   Neha Singh|             28|
|        111|   Ravi Kumar|           NULL|
|        102|  Priya Reddy|            102|
+-----------+-------------+---------------+



In [114]:
updated_customer_usage_billing_df = updated_customer_usage_full_df \
    .join(
        payments_clean_df.withColumnRenamed("data_quality_status", "payment_dq_status"),
        on="customer_id", how="left"
    )

updated_customer_usage_billing_df.write.mode("overwrite") \
    .partitionBy("usage_month") \
    .parquet(gold_path)

In [116]:
before_count = usage_clean_df.count()
after_count = updated_usage_df.count()
print("Usage count before incremental load:", before_count)
print("Usage count after incremental load:", after_count)
print("New records added:", after_count - before_count)

Usage count before incremental load: 15
Usage count after incremental load: 20
New records added: 5


In [117]:
# Part 10: Final Gold Reports

from pyspark.sql.functions import countDistinct, avg, sum, when, col

In [118]:
payments_for_join_df = payments_clean_df.withColumnRenamed(
    "data_quality_status", "payment_dq_status"
).withColumnRenamed("bill_month", "usage_month")

customer_month_df = updated_usage_df.withColumnRenamed(
    "data_quality_status", "usage_dq_status"
).join(
    payments_for_join_df,
    on=["customer_id", "usage_month"],
    how="left"
)

gold_customer_usage_df = customers_clean_df.withColumnRenamed(
        "data_quality_status", "customer_dq_status"
    ).join(
        plans_flat_df.withColumnRenamed("data_quality_status", "plan_dq_status"),
        on="plan_id", how="left"
    ).join(
        customer_month_df, on="customer_id", how="left"
    )

# Re-derive over-usage and churn risk on the aligned data
gold_customer_usage_df = gold_customer_usage_df.withColumn(
    "over_usage_gb", col("data_used_gb") - col("data_limit_gb")
).withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0, "Yes").otherwise("No")
).withColumn(
    "churn_risk",
    when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success") |
        (col("payment_status").isNull()),
        "High Risk"
    )
    .when(col("data_used_gb") < 15, "Medium Risk")
    .otherwise("Low Risk")
)

gold_customer_usage_df.show(truncate=False)

+-----------+-------+-------------+---------+-----------+---+------+--------+------------------+------------+-----------+-------------+---------------+------------+-------------+--------------+-------------------+--------+------------+------------+---------+---------------+----------+-----------+------------+--------------+-----------------+-------------+---------------+----------+
|customer_id|plan_id|customer_name|city     |state      |age|gender|status  |customer_dq_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |plan_dq_status|usage_month        |usage_id|data_used_gb|call_minutes|sms_count|usage_dq_status|payment_id|amount_paid|payment_mode|payment_status|payment_dq_status|over_usage_gb|over_usage_flag|churn_risk|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------------+------------+-----------+-------------+---------------+------------+-------------+--------------+-------------------+--------+--------

In [119]:
customer_usage_summary_report_df = gold_customer_usage_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "usage_month",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_flag",
    "amount_paid",
    "payment_status",
    "churn_risk"
)

customer_usage_summary_report_df.show(truncate=False)

customer_usage_summary_report_df.write.mode("overwrite").parquet("gold/customer_usage_summary")

+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|customer_id|customer_name|city     |plan_name   |usage_month        |data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status|churn_risk|
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-03-01 00:00:00|52          |50           |Yes            |NULL       |NULL          |High Risk |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |50           |No             |499        |Success       |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-01-01 00:00:00|45          |50           |No             |499        |Success       |Low Risk  |
|102        |Priya Reddy  |Bangalore|Smart Plus  |2026-03-01 00:00:00|38          |75   

In [120]:
plan_performance_df = gold_customer_usage_df.groupBy("plan_name").agg(
    countDistinct("customer_id").alias("total_customers"),
    sum("data_used_gb").alias("total_data_usage"),
    avg("data_used_gb").alias("average_data_usage"),
    sum(
        when(col("payment_status") == "Success", col("amount_paid")).otherwise(0)
    ).alias("total_revenue")
)
plan_performance_df.show()
plan_performance_df.write.mode("overwrite").parquet("gold/plan_performance_report")

+------------+---------------+----------------+------------------+-------------+
|   plan_name|total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
|        NULL|              2|            NULL|              NULL|            0|
| Smart Basic|              3|             368| 52.57142857142857|         2495|
|Budget Saver|              2|              22|              11.0|            0|
| Premium Max|              2|             318|              63.6|         2398|
|  Smart Plus|              3|             162|              32.4|         3196|
+------------+---------------+----------------+------------------+-------------+



In [121]:
city_revenue_df = gold_customer_usage_df.groupBy("city").agg(
    countDistinct("customer_id").alias("total_customers"),
    sum(
        when(col("payment_status") == "Success", col("amount_paid")).otherwise(0)
    ).alias("total_revenue"),
    avg(
        when(col("payment_status") == "Success", col("amount_paid"))
    ).alias("average_payment")
)
city_revenue_df.show()
city_revenue_df.write.mode("overwrite").parquet("gold/city_revenue_report")

+---------+---------------+-------------+---------------+
|     city|total_customers|total_revenue|average_payment|
+---------+---------------+-------------+---------------+
|Bangalore|              2|         2097|          699.0|
|    Kochi|              1|         1199|         1199.0|
|  Chennai|              1|          998|          499.0|
|   Mumbai|              2|            0|           NULL|
|     Pune|              1|          799|          799.0|
|    Delhi|              2|         1998|          999.0|
|Hyderabad|              3|          998|          499.0|
+---------+---------------+-------------+---------------+



In [122]:
churn_risk_report_df = gold_customer_usage_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "payment_status",
    "status",
    "churn_risk"
).distinct()
churn_risk_report_df.show()
churn_risk_report_df.write.mode("overwrite").parquet("gold/churn_risk_report")

+-----------+-------------+---------+------------+--------------+--------+----------+
|customer_id|customer_name|     city|   plan_name|payment_status|  status|churn_risk|
+-----------+-------------+---------+------------+--------------+--------+----------+
|        105|   Farhan Ali|    Delhi| Premium Max|       Pending|  Active| High Risk|
|        108|   Meera Nair|    Kochi| Premium Max|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        110|  Nisha Reddy|    Delhi|  Smart Plus|       Success|  Active|  Low Risk|
|        105|   Farhan Ali|    Delhi| Premium Max|       Success|  Active|  Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|  Low Risk|
|        106|   Neha Singh|     Pune|  Smart Plus|       Success|  Active|  Low Risk|
|        109|    Kiran Rao|Bangalore| Smart Basic|       Success|  Active|  Low Risk|
|        107|  Arjun Verma|Hyderabad|Budget Saver|    

In [123]:
over_usage_report_df = gold_customer_usage_df.filter(
    col("over_usage_flag") == "Yes"
).select(
    "customer_id",
    "customer_name",
    "plan_name",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_gb"
)
over_usage_report_df.show()
over_usage_report_df.write.mode("overwrite").parquet("gold/over_usage_report")

+-----------+-------------+-----------+------------+-------------+-------------+
|customer_id|customer_name|  plan_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+-----------+------------+-------------+-------------+
|        101| Rahul Sharma|Smart Basic|          52|           50|            2|
|        104|  Sneha Patel|Smart Basic|          60|           50|           10|
|        104|  Sneha Patel|Smart Basic|          58|           50|            8|
|        104|  Sneha Patel|Smart Basic|          55|           50|            5|
+-----------+-------------+-----------+------------+-------------+-------------+

